# Hello, from mercury-dataschema 👋

<img src="images/dataschema.png" width="400">

Modern data workflows rely heavily on **dataframes** as the core abstraction for analysis, feature engineering, and machine learning pipelines. However, one common challenge across these workflows is **understanding and managing data types** consistently. Columns may contain mixed values, implicit formats, or ambiguous types that require manual inspection and transformation before the data can be safely used.

[mercury-dataschema](https://github.com/BBVA/mercury-dataschema) addresses this problem by introducing a lightweight layer on top of your dataframes that provides **automatic or manual** type detection and structured schema management. This library is used in other mercury modules.

At its core, DataSchema performs two key functions:

1. **Automatic Type Detection**  
   It analyzes dataframe columns and automatically infers their semantic types (such as categorical variables, numerical values, text fields, dates, etc.). This removes the need for repetitive manual inspection and reduces the risk of inconsistent assumptions about the data.

2. **A Schema Wrapper Around DataFrames**  
   DataSchema wraps your dataframe with a schema object that **tracks metadata and column types while remaining fully compatible with existing dataframe operations**. This design allows you to continue using familiar libraries and tools while benefiting from a richer understanding of the dataset structure.

Think about it as a **thin, non-intrusive layer** that allows your validated column types to have a lifecycle that keeps consistency across
different releases of the same file.

In this notebook, we will explore how DataSchema works and demonstrate the main capabilities it provides for **automated schema detection and dataframe-aware data processing**.


In [ ]:
from mercury.dataschema import DataSchema
from mercury.dataschema.anonymize import Anonymize


## Getting a dataset from seaborn examples

To demonstrate the capabilities of **DataSchema**, we start by loading a sample dataset from the **Seaborn example datasets**. Seaborn provides a convenient collection of small, well-known datasets.

We import seaborn just in case just to load the tips dataset to play with it.

We pip install it first.


In [ ]:
!pip install seaborn

In [ ]:
import seaborn as sns


We change the types of the strings to string.

In [ ]:
tips = sns.load_dataset('tips')
tips['sex'] = tips['sex'].astype(str)
tips['smoker'] = tips['smoker'].astype(str)
tips['day'] = tips['day'].astype(str)
tips['time'] = tips['time'].astype(str)

tips


## Automated type detection

One of the main features of **DataSchema** is its ability to automatically infer the semantic type of each column in a dataframe. 

The method `.generate()` creates, for each column, an object of class **`Feature`**. This object abstracts the column’s properties and metadata, allowing it to be handled in a **consistent way regardless of the underlying data type**. Instead of working directly with raw dataframe columns, downstream tools can operate on these `Feature` objects.

This abstraction is a key design principle used across many **Mercury** packages. By standardizing how features are represented, different libraries in the ecosystem can interact with datasets in a uniform and predictable way.

As you may notice from the warning, DataSchema may sometimes classify a numeric column as **categorical**. For example, an integer column containing only two unique values may be interpreted as a categorical variable rather than a continuous numeric feature. This behavior is intentional and based on heuristics designed to capture the semantic meaning of the data.

These heuristics can be **customized and controlled** if needed.

*See the documentation for details.*


In [ ]:
schema = DataSchema().generate(tips)

The method `.generate` generates for each of the columns an object of class Feature that allows abstracting its details
and using it in the same way across types.

This is how many mercury packages work.

As you can see in the previous warning, it treats an integer variable as categorical because it has only two values. This behavior can be controlled 

  * [see documentation](https://bbva.github.io/mercury-dataschema/)

In [ ]:
schema.feats

## Data anonymization

Another useful capability provided by **DataSchema** is the ability to **anonymize datasets while preserving their structural properties**. This is particularly helpful when sharing data for demonstrations, testing, or documentation, where the goal is to keep the dataset realistic without exposing sensitive or identifiable information.

Using the schema generated in the previous step, DataSchema knows the **semantic type of each feature** in the dataframe. This allows the library to apply anonymization strategies that are **appropriate for each feature type**. For example, categorical variables, numerical variables, dates, or text fields can each be transformed in a way that maintains the general structure and statistical characteristics of the data.

The `anonymize` function leverages the **Feature objects produced by the schema** to generate a new dataframe where the original values are replaced with anonymized ones, while keeping the dataset consistent and usable for experimentation. Because the transformation is guided by the schema, the resulting dataset remains compatible with typical analytical workflows.

This makes it possible to **share or document datasets safely**, while still preserving the properties needed to demonstrate how tools and pipelines behave on realistic data.

In [ ]:
anon = Anonymize()

### Cryptographic anonymization

In some cases, instead of generating realistic synthetic values, it is preferable to **replace the values of a column with a secure identifier**. DataSchema provides a mechanism to transform column values into **cryptographically secure keys of a configurable length**, effectively masking the original information while preserving uniqueness and consistency.

The anonymization process is **deterministic** and controlled by a key defined by the user. This key determines how the anonymized values are generated internally. It can be set with:

```python
anon.set_key("Mickey Mouse")
````

Because the algorithm uses this key as part of the transformation, the same original value will always produce the **same anonymized token** when the same key is used. This allows anonymization to remain **reproducible across runs**, which can be important when anonymizing datasets that need to maintain stable relationships between values.

At the same time, the transformation is **cryptographically secure**, meaning that the original values cannot be easily reconstructed from the anonymized ones. The length of the generated identifiers can also be configured depending on the needs of the dataset.

In [ ]:
anon.set_key('Mickey Mouse')

In [ ]:
anon.anonymize_list_any_type(list(tips['total_bill']))[0:10]

## Same example with shorter digest length

We run the same example with 12 bit digest (2 base-64 digits).

In [ ]:
anon = Anonymize(digest_bits = 12)

In [ ]:
anon.set_key('Mickey Mouse')

In [ ]:
anon.anonymize_list_any_type(list(tips['total_bill']))[0:10]